# Lab 6: Principal Component Analysis

<a target="_blank" href="https://colab.research.google.com/github/drchadvidden/courseMaterials/blob/main/UnsupervisedLearning/Labs/Lab%206/Lab_6.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lab Instructions

Run each of the coding cells. For tutorial example cells, understand the commands and check that the outputs make sense. For exercise cells, write your own code where indicated to generate the correct output. Give text explanations where indicated.

### Submission:
Complete the following notebook in order. Once done, save the notebook, print the file as a .pdf, and upload the resulting file to the Canvas course assignment.

### Rubric:
15 total points, 5 points to running tutorial example cells and saving outputs, 10 points for completing exercises.

### Deadline:
Tuesday at midnight after the lab is assigned.

# Tutorial: PCA Explained

In this tutorial we illustrate how to compute and explore **Principal Component Analysis (PCA)** using Python. PCA is a dimensionality reduction technique that transforms a dataset with many correlated variables into a smaller set of new variables called **principal components**, which capture the main directions of variation in the data.

Suppose we have a dataset with $n$ observations and $p$ variables. After centering (and often scaling) the data, we represent it as a matrix

$$
X \in \mathbb{R}^{n \times p}.
$$

The **covariance matrix** of the variables is

$$
\Sigma = \frac{1}{n-1} X^T X.
$$

PCA finds directions in feature space along which the data varies the most. These directions are given by the **eigenvectors** of the covariance matrix:

$$
\Sigma v_i = \lambda_i v_i
$$

where

- $v_i$ is the $i$-th principal component direction  
- $\lambda_i$ is the variance explained by that component

The eigenvalues are ordered

$$
\lambda_1 \ge \lambda_2 \ge \cdots \ge \lambda_p
$$

so that the **first principal component** captures the largest possible variance in the data, the second captures the largest remaining variance subject to being orthogonal to the first, and so on.

Each observation can be **projected onto the principal components** to obtain new coordinates called **principal component scores**:

$$
Z = X V
$$

where $V = [v_1, v_2, \dots, v_p]$ contains the eigenvectors as columns.

In practice, PCA is usually computed using the **Singular Value Decomposition (SVD)** of the data matrix:

$$
X = U \Sigma V^T
$$

The columns of $V$ correspond to the principal component directions, and the singular values determine the variance explained by each component.

PCA is widely used for:

- **Dimensionality reduction**
- **Visualization of high-dimensional data**
- **Noise reduction**
- **Feature extraction before clustering or machine learning**

In this tutorial we will apply PCA to **customer segmentation** and the famous **MNIST handwritten digit dataset**, which contains thousands of images of digits. Each image has $28 \times 28 = 784$ pixel features. PCA will allow us to explore the main directions of variation in these images and visualize them in just a few dimensions.

## PCA for dataset exploration

Here we revisit the mall customer dataset from past homeworks.

In [ ]:
import pandas as pd

# this file is also hosted on Kaggle: https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python
url = 'https://gist.githubusercontent.com/pravalliyaram/5c05f43d2351249927b8a3f3cc3e5ecf/raw/8bd6144a87988213693754baaa13fb204933282d/Mall_Customers.csv'
df = pd.read_csv(url)

print(df.head())
print(df.info())
print(df.describe())
print(df["Gender"].value_counts())

In this section, we explore how Principal Component Analysis (PCA) can help us understand customer structure in a high-dimensional dataset.

We focus on three quantitative variables:

- Age  
- Annual Income (k$)  
- Spending Score (1–100)  

Our goals are to:

1. Standardize the features (since PCA is variance-based).
2. Compute principal components.
3. Examine how much variance each component explains.
4. Interpret the loadings to understand what each principal direction represents.
5. Connect PCA back to the covariance matrix and total variance.

Conceptually, PCA finds new orthogonal directions that capture the maximum possible variance in the data. The first principal component explains the largest possible variance, the second explains the largest remaining variance (subject to orthogonality), and so on.

We will analyze:

- The explained variance ratio (how much information each component retains),
- A scree plot, which is a line plot of the explained variance ratio versus principal component number. It helps us determine how many components are worth keeping by showing how quickly the variance decreases. An “elbow” in the plot often suggests a natural cutoff for dimensionality reduction.
- The loading vectors (how original features combine to form each component),
- The covariance matrix and its total variance,
- The relationship between total variance and the variance explained by PCA.

This connects linear algebra (eigenvalues, covariance matrices) directly to practical data exploration and segmentation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# -----------------------
# 1. Select features
# -----------------------
features = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
X = df[features].copy()

# -----------------------
# 2. Standardize
# -----------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -----------------------
# 3. PCA
# -----------------------
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Explained variance
explained_var = pca.explained_variance_ratio_
print("Explained variance ratio:", explained_var)
print("Cumulative explained variance:", np.cumsum(explained_var))

# -----------------------
# 4. Scree Plot
# -----------------------
plt.figure()
plt.plot(range(1, len(explained_var)+1), explained_var, marker='o')
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("Scree Plot")
plt.show()

# -----------------------
# 5. Loadings
# -----------------------
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f"PC{i+1}" for i in range(len(features))],
    index=features
)
print("\nPCA Loadings:")
print(loadings)

# -----------------------
# 6. Covariance Matrix
# -----------------------

cov_matrix = np.cov(X_scaled, rowvar=False)
print("\nCovariance Matrix:")
print(pd.DataFrame(cov_matrix, index=features, columns=features))

# -----------------------
# 7. Total Variance
# -----------------------

total_variance = np.trace(cov_matrix)
print("\nTotal Variance (trace of covariance matrix):", total_variance)

# -----------------------
# 8. PCA Variance Explained
# -----------------------

print("\nPCA Explained Variance (eigenvalues):")
print(pca.explained_variance_)

print("\nPCA Explained Variance Ratio:")
print(pca.explained_variance_ratio_)

print("\nSum of PCA Explained Variance:", pca.explained_variance_.sum())


## Visualizing Principal Components

After computing PCA, it is helpful to visualize how the principal components relate to the data and the original features as seen in the **above PCA loadings**.

### 1. PCA Biplot (PC Space)
- The first plot shows customers projected onto the **first two principal components** (PC1 vs PC2).  
- Each point represents a customer in this reduced 2D space.  
- The red arrows indicate the **loadings**, showing how strongly each original feature contributes to each principal component.  
- This helps interpret what PC1 and PC2 capture in terms of **Age, Annual Income, and Spending Score**.  
- Directions of the arrows reveal whether features are aligned or opposed in the component space.

### 2. PCs in Original Feature Space
- The second plot maps the principal components back onto the **standardized original features** (here Age vs Annual Income).  
- Each point is a customer in the original feature plane.  
- The red arrows represent the **directions of PC1 and PC2** in terms of the original variables.  
- This illustrates how the PCs summarize the original data and highlights which features drive the main sources of variation.  
- Together, these plots connect the abstract principal components to interpretable, real-world customer attributes, facilitating segmentation and pattern discovery.

In [ ]:
# -----------------------
# 9. PCA Biplot: PCs vs Original Features
# -----------------------

plt.figure(figsize=(8,6))

# Scatter of the projected customers in PC space
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.6)

# Draw loading vectors for original features
for i, feature in enumerate(features):
    plt.arrow(0, 0,
              loadings.loc[feature, "PC1"]*3,  # scale for visibility
              loadings.loc[feature, "PC2"]*3,
              color='r', alpha=0.8, head_width=0.1)
    plt.text(loadings.loc[feature, "PC1"]*3,
             loadings.loc[feature, "PC2"]*3,
             feature, color='r', fontsize=12)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA Biplot: Customer Projection with Feature Loadings")
plt.axhline(0, color='grey', linewidth=0.5)
plt.axvline(0, color='grey', linewidth=0.5)
plt.grid(True)
plt.show()

# -----------------------
# 10. Plot PCs in Original Feature Space
# -----------------------
import seaborn as sns

# Standardize features for plotting
X_std = pd.DataFrame(X_scaled, columns=features)

plt.figure(figsize=(8,6))

# Scatter each customer in the original feature space (PC1 vs PC2)
# We'll plot the first two features (Age vs Annual Income) for visualization
sns.scatterplot(x=X_std["Age"], y=X_std["Annual Income (k$)"], alpha=0.6)

# Overlay PC1 and PC2 directions
for i, pc in enumerate(["PC1", "PC2"]):
    plt.arrow(0, 0,
              loadings.loc["Age", pc]*3,   # scale for visibility
              loadings.loc["Annual Income (k$)", pc]*3,
              color='r', alpha=0.8, head_width=0.1)
    plt.text(loadings.loc["Age", pc]*3,
             loadings.loc["Annual Income (k$)", pc]*3,
             pc, color='r', fontsize=12)

plt.xlabel("Standardized Age")
plt.ylabel("Standardized Annual Income (k$)")
plt.title("PC1 & PC2 Directions in Original Feature Space")
plt.axhline(0, color='grey', linewidth=0.5)
plt.axvline(0, color='grey', linewidth=0.5)
plt.grid(True)
plt.show()

## Clustering Customers and Interpreting in PC Space

In this section, we apply **KMeans clustering** to the full standardized dataset to identify groups of similar customers.  

### Key points:

1. **Clustering on full data:**  
   - We use all three features (Age, Annual Income, Spending Score) to form clusters.  
   - This ensures that important variance along all dimensions is captured, not just the first two PCs as would happen if dimension reduction was performed before clustering.

2. **Visualizing clusters on PC1–PC2:**  
   - The scatter plot shows customers projected onto the **first two principal components** for easy visualization.  
   - Each color represents a cluster found in the full feature space.  
   - Cluster centroids are projected onto PC space to indicate the typical position of each group.

3. **Interpreting clusters using PC loadings:**  
   - The **loadings** of the original features tell us what each principal component represents:  
     - PC1 might contrast Age vs Spending Score, PC2 could reflect Annual Income, etc.  
   - By examining where clusters lie in PC1–PC2 space and the directions of the loadings, we can **infer the characteristic patterns of each cluster in terms of the original features**.  
   - For example, a cluster located along the positive direction of PC2 may correspond to high-income customers, while its PC1 coordinate shows the Age/Spending Score balance.

4. **Cluster summaries:**  
   - After clustering, we compute mean and standard deviation for each original variable within each cluster.  
   - This directly links the abstract clusters back to **real-world customer attributes**, making the segmentation interpretable.

Overall, this approach combines **unsupervised learning, dimensionality reduction, and interpretable visualization** to reveal patterns in customer data.

In [ ]:
# -----------------------
# 11. Clustering on Full Standardized Data, Visualize on PCs
# -----------------------
from sklearn.cluster import KMeans

# Cluster on full standardized features
kmeans_full = KMeans(n_clusters=5, random_state=42)
full_cluster_labels = kmeans_full.fit_predict(X_scaled)

# Project customers onto first 2 PCs for visualization
X_pca_2d = X_pca[:, :2]

# Create DataFrame for plotting
df_clusters_pca = pd.DataFrame(X_pca_2d, columns=["PC1", "PC2"])
df_clusters_pca["Cluster"] = full_cluster_labels

# -----------------------
# 12. Visualize Clusters in PC Space
# -----------------------
plt.figure(figsize=(8,6))
colors = ["red", "blue", "green", "orange", "purple"]

for i in range(5):
    cluster_data = df_clusters_pca[df_clusters_pca["Cluster"] == i]
    plt.scatter(cluster_data["PC1"], cluster_data["PC2"],
                color=colors[i], label=f"Cluster {i+1}", alpha=0.6)

# Plot cluster centroids (projected onto first 2 PCs)
centroids_full_pca = kmeans_full.cluster_centers_ @ pca.components_[:2].T
plt.scatter(centroids_full_pca[:,0], centroids_full_pca[:,1],
            marker="X", s=200, color="black", label="Centroids")

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("KMeans Clusters (Full Data) Visualized on First Two PCs")
plt.legend()
plt.axhline(0, color='grey', linewidth=0.5)
plt.axvline(0, color='grey', linewidth=0.5)
plt.grid(True)
plt.show()

# -----------------------
# 13. Cluster Summary
# -----------------------
print("Cluster sizes:")
print(df_clusters_pca["Cluster"].value_counts())

# Add cluster labels to original DataFrame
df_orig_clusters = df.copy()
df_orig_clusters["Cluster"] = full_cluster_labels

# Compute cluster-wise summaries for Age, Income, Spending Score
cluster_summary = df_orig_clusters.groupby("Cluster").agg(
    Count=("Age", "count"),
    Age_Mean=("Age", "mean"),
    Age_Std=("Age", "std"),
    Income_Mean=("Annual Income (k$)", "mean"),
    Income_Std=("Annual Income (k$)", "std"),
    Spending_Mean=("Spending Score (1-100)", "mean"),
    Spending_Std=("Spending Score (1-100)", "std")
).reset_index()

print("Cluster summaries:")
print(cluster_summary)

# Tutorial: PCA for MNIST

Here we explore PCA on the famous handwriting dataset, MNIST.  

- MNIST is widely used for teaching and benchmarking machine learning algorithms.  
- Original dataset reference: **Y. LeCun, C. Cortes, and C. J. Burges, "MNIST handwritten digit database," 1998.**  
  - [MNIST Dataset Online](http://yann.lecun.com/exdb/mnist/)  

This tutorial will demonstrate how to **reduce dimensionality, visualize, and interpret patterns** in the dataset using PCA.

## Exploring the MNIST Dataset

Before applying PCA, it's helpful to **inspect the data** to understand what we're working with.  

- MNIST consists of **28×28 grayscale images** of handwritten digits (0–9).  
- Each image is flattened into a **784-dimensional vector** in the dataset.  
- `X` contains the pixel values, and `y` contains the corresponding digit labels.  
- Visualizing a few examples allows students to **see the variation in handwriting** and get a sense of the patterns PCA will later capture.  

The code below loads MNIST and displays a small set of sample digits.

In [ ]:
# -----------------------
# 1. Load MNIST and Visualize Sample Digits
# -----------------------
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
import numpy as np

# Load MNIST dataset
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X, y = mnist.data, mnist.target.astype(int)

# Take a small subset for visualization
subset_size = 12
X_sample = X[:subset_size]
y_sample = y[:subset_size]

# Plot a few sample digits
plt.figure(figsize=(10,4))
for i in range(subset_size):
    plt.subplot(2, 6, i+1)
    plt.imshow(X_sample[i].reshape(28,28), cmap="gray")
    plt.title(f"{y_sample[i]}")
    plt.axis("off")
plt.suptitle("Sample MNIST Digits")
plt.show()

## PCA on MNIST (Subset) with Variance Analysis and Visualization

Now that we've explored the raw MNIST images, we apply **Principal Component Analysis (PCA)** to reduce dimensionality and uncover the main directions of variance in the data.

- We take a **subset of 5000 images** to keep computations fast.  
- All pixel values are **standardized** to ensure each pixel contributes equally.  
- We compute the **first 10 principal components** to analyze variance and prepare visualizations.  
- The **scree plot** shows both individual and cumulative variance explained, helping to determine how many PCs capture most of the information.  
- We then visualize the data in **2D (PC1 vs PC2)** and **3D (PC1 vs PC2 vs PC3)**, coloring points by their digit label to see natural clustering patterns.  

This step allows students to see how **high-dimensional image data can be summarized in a low-dimensional space**, while retaining most of the variance.

In [ ]:
# -----------------------
# 2. PCA on MNIST (Subset) + Variance Info
# -----------------------
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # for 3D plotting
import pandas as pd

# Use a subset for faster computation
subset_size = 5000
X_small = X[:subset_size]
y_small = y[:subset_size]

# Standardize pixel values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_small)

# Compute PCA for first 5 components (for cumulative variance and 3D plotting)
pca = PCA(n_components=10)
X_pca_all = pca.fit_transform(X_scaled)

# Create a DataFrame of PCs for easy plotting
df_pca = pd.DataFrame(X_pca_all, columns=[f"PC{i+1}" for i in range(10)])
df_pca["Digit"] = y_small

# -----------------------
# Explained variance and scree plot
# -----------------------
explained_var = pca.explained_variance_ratio_
cum_var = explained_var.cumsum()
print("Explained variance ratio (PC1-PC10):", explained_var)
print("Cumulative explained variance (PC1-PC10):", cum_var)
plt.figure(figsize=(8,6))
plt.plot(range(1, 11), explained_var, marker='o', linestyle='-', label="Individual PC")
plt.plot(range(1, 11), cum_var, marker='s', linestyle='--', label="Cumulative")
plt.xticks(range(1,11))
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("Scree Plot for First 10 Principal Components (MNIST Subset)")
plt.legend()
plt.grid(True)
plt.show()

# -----------------------
# 2D Scatter plot (PC1 vs PC2)
# -----------------------
plt.figure(figsize=(10,8))
scatter = plt.scatter(df_pca["PC1"], df_pca["PC2"], c=df_pca["Digit"], cmap="tab10", alpha=0.6, s=10)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("MNIST Digits Projected onto First Two Principal Components (2D)")
plt.colorbar(scatter, ticks=range(10), label="Digit Label")
plt.grid(True)
plt.show()

# -----------------------
# 3D Scatter plot (PC1 vs PC2 vs PC3)
# -----------------------
fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection='3d')
p = ax.scatter(df_pca["PC1"], df_pca["PC2"], df_pca["PC3"], c=df_pca["Digit"], cmap="tab10", alpha=0.6, s=10)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.set_title("MNIST Digits Projected onto First Three Principal Components (3D)")
fig.colorbar(p, ticks=range(10), label="Digit Label")
plt.show()

## Visualizing PCA Feature Contributions

After computing PCA, it is often useful to understand **what each principal component is capturing** in the original feature space.  

- Each principal component is a **linear combination of all 784 pixels**.  
- By reshaping the **component vector into a 28×28 image**, we can see **which pixels contribute most positively or negatively** to that PC.  
- Red areas indicate pixels that **increase the component value**, blue areas indicate pixels that **decrease it**.  
- Visualizing the **top 5 PCs** helps students interpret how PCA identifies patterns in handwritten digits, such as stroke orientation, loops, or line positions.  

The code below plots the first five principal components as images to highlight these contributions.

In [ ]:
# -----------------------
# 3. PCA Feature Contributions: Visualize Top Pixels
# -----------------------
import numpy as np

# Function to plot a PCA component as an image
def plot_pca_component(pc_index, pca, figsize=(5,5)):
    plt.figure(figsize=figsize)
    component_image = pca.components_[pc_index].reshape(28,28)
    plt.imshow(component_image, cmap='seismic', vmin=-np.max(np.abs(component_image)), vmax=np.max(np.abs(component_image)))
    plt.colorbar(label="Contribution")
    plt.title(f"PCA Component {pc_index+1} (Pixel Contributions)")
    plt.axis("off")
    plt.show()

# Plot top 5 PCs
for i in range(5):
    plot_pca_component(i, pca)

# Exercise(s): PCA





## Exercise 1: Homework check

### Tasks:
1. Verify your PCA findings for problems 19, 20 and 22 of the homework.
2. If you want, try to visualize the geometry of PCA for each.

In [ ]:
# Write your code for the exercise here!

### Explain your findings here:




## Exercise 2: PCA via Eigen Decomposition

Use python to reproduce the PCA mathematics for problem 20 of the homework. Your goal is to compute a **single size index** that captures most of the variation in the data.

### Tasks / Approach

1. **Center the data** by subtracting the mean of each feature (weight, height): `X_centered = X - X.mean(axis=0)`

2. **Compute the covariance matrix** using the centered data: `cov_matrix = np.cov(X_centered, rowvar=False)`

3. **Compute eigenvalues and eigenvectors** of the covariance matrix: `eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)`

4. **Identify the first principal component** as the eigenvector corresponding to the largest eigenvalue. This represents the direction of maximal variation.

5. **Project the data onto the first principal component** to obtain a **size index**: `size_index = X_centered @ eigenvectors[:, 0]`

6. **Compute the proportion of variance explained** by the first principal component: `explained_variance_ratio = eigenvalues / eigenvalues.sum()`

7. **Repeat PCA without scaling the data** to explore how the results and the size index change compared to the scaled version.


**Goal:** understand PCA from first principles using eigen decomposition, and verify how the **size index** combines weight and height into a single measure of overall size. Show important calulations and verify findings.

In [ ]:
# Write your code for the exercise here!

### Explain your findings here:




## Exercise 3: PCA and Clustering on Wholesale Customers Dataset

In this exercise, you will **repeat the PCA and clustering workflow** you performed for the previous customer segmentation dataset, but now on the **Wholesale Customers dataset** from the UCI repository.

### Problem Statement
The dataset contains spending amounts for various product categories from different wholesale customers. Your goal is to:

1. **Explore the data** and understand the distribution of spending across categories.
2. **Apply PCA** to reduce dimensionality and identify the main patterns of variance in customer spending.
3. **Visualize** the data in 2D and 3D PC space, interpreting how different spending categories contribute to the principal components.
4. **Cluster customers** using KMeans on the full standardized dataset, then visualize cluster assignments in PC space.
5. **Summarize clusters** over original variables to interpret typical spending behaviors for each group.

### Tasks

1. **Data Exploration**
   - Examine the first few rows of the dataset.
   - Check for missing values and basic statistics of numeric columns.
   
2. **Feature Selection and Standardization**
   - Select relevant numeric features: `Fresh`, `Milk`, `Grocery`, `Frozen`, `Detergents_Paper`, `Delicassen`.
   - Standardize the features before applying PCA.
   
3. **PCA Analysis**
   - Compute principal components (at least the first 5).
   - Plot a **scree plot** showing explained and cumulative variance.
   - Visualize the data in **2D (PC1 vs PC2)** and **3D (PC1 vs PC2 vs PC3)**.
   - Interpret which product categories contribute most to each PC using **loadings**.

4. **Clustering**
   - Perform **KMeans clustering** on the full standardized data.
   - Visualize clusters in PC space with cluster centroids.
   - Summarize cluster characteristics over the original numeric features.

Use the code framework from the previous customer segmentation lab, adapting it for this dataset.  


In [ ]:
# Write your code for the exercise here!

import pandas as pd

df_wholesale = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv")
df_wholesale.head()

### Explain your findings here:




## Exercise 4: Clustering MNIST Digits Using PCA

Extend the PCA analysis on the MNIST subset to explore **unsupervised clustering**. Use the PCA-transformed data to cluster the digits and compare the clusters to the true labels.

### Tasks

1. Use the PCA results (`df_pca`) containing the first few principal components. Start with the **first 5 PCs**.

2. Select a **number of clusters** equal to the number of digit classes (10).  

3. Apply **KMeans clustering** to the PCA-transformed data (using 5 PCs).  

4. **Visualize the clusters** on the first two and three principal components (2D and 3D scatter plots).  

5. **Compare the clusters to the true digit labels** using a contingency table. Explore correct and incorrect classifications by inspecting images.

6. **Clustering analyses:**  
   - *Compute cluster centroids* in PC space and see which clusters are near others.  
   - *Visualize cluster centroids* as images to see prototype cluster digits.
   - *Measure clustering quality* using metrics such as WCSS or silhouette score.  

7. **Repeat the clustering and evaluation** using **10 PCs** and then **15 PCs**. Compare accuracy results across 5, 10, and 15 PCs to see how dimensionality affects clustering quality. Also explore variance captured for each.

8. **Cluster on the original feature space** with 784 (scaled) pixel features and compare to findings in task 7.

In [ ]:
# Write your code for the exercise here!

### Explain your findings here:




# HTML Export Code

In [ ]:
# code to export notebook as .html for Canvas upload

from google.colab import drive
from google.colab import files

drive.mount('/content/drive')

notebook_name = "Lab_6"
!cp "/content/drive/MyDrive/Colab Notebooks/DSC 430/{notebook_name}.ipynb" /content/
!jupyter nbconvert --to html "/content/{notebook_name}.ipynb"
files.download(f"/content/{notebook_name}.html")

